In [1]:
import os
import sys
from typing import Dict, List, Any

# LangChain, Hugging Face, and Vector Store imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import torch
import gradio as gr

C:\Users\HP\AppData\Local\Temp\ipykernel_16952\2858362579.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# 1. Local Sentence-Transformers Embedding Model (No API Key Required)
print("Loading SentenceTransformer model ('all-MiniLM-L6-v2')...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"} # Change to "cuda" if you have an NVIDIA GPU
)

# 2. Local Open-Source LLM (Flan-T5 for lightweight, local CPU inference)
print("Loading local Hugging Face LLM ('google/flan-t5-base')...")
model_id = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# FIX: Explicitly specify the task so the pipeline accepts the model object
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.01
)

llm = HuggingFacePipeline(pipeline=pipe)
print("Local AI models initialized!")

Loading SentenceTransformer model ('all-MiniLM-L6-v2')...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading local Hugging Face LLM ('google/flan-t5-base')...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCaus

Local AI models initialized!


In [3]:
PDF_FILE_PATH = "OB.pdf"  # Ensure OB.pdf is in the same directory as this notebook

def build_or_load_vector_db():
    index_path = "faiss_ob_course_index_local"

    if os.path.exists(index_path):
        print("Loading existing local FAISS index from disk...")
        vector_db = FAISS.load_local(
            index_path, 
            embeddings, 
            allow_dangerous_deserialization=True
        )
    else:
        if not os.path.exists(PDF_FILE_PATH):
            raise FileNotFoundError(f"'{PDF_FILE_PATH}' not found! Place 'OB.pdf' in the current working directory.")
            
        print("Processing PDF and building new local FAISS index...")
        # 1. Load Documents & Extract Text
        loader = PyPDFLoader(PDF_FILE_PATH)
        documents = loader.load()

        # 2. Chunking Strategy
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=700,
            chunk_overlap=100
        )
        chunks = text_splitter.split_documents(documents)

        # 3. Local Sentence-Transformers Vector Embeddings & Vector Database Construction
        vector_db = FAISS.from_documents(chunks, embeddings)
        vector_db.save_local(index_path)
        print("FAISS vector store successfully constructed and saved.")

    return vector_db

# Build/Load Vector Database and Retriever
vector_db = build_or_load_vector_db()
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

Loading existing local FAISS index from disk...


In [4]:
# System Prompt with Anti-Hallucination & Citation Rules
template = """You are an academic assistant for an Organizational Behavior course.
Answer the question using ONLY the provided context below.

STRICT RULES:
1. If the required information is NOT available in the context, respond EXACTLY with:
   "I couldn't find this information in the provided knowledge base."
2. DO NOT use external knowledge, speculate, or invent details.

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate.from_template(template)

def format_docs_with_metadata(docs):
    formatted = []
    for doc in docs:
        page = doc.metadata.get("page", 0) + 1  # 1-indexed page display
        source = os.path.basename(doc.metadata.get("source", "OB.pdf"))
        formatted.append(f"--- Document: {source} (Page {page}) ---\n{doc.page_content}")
    return "\n\n".join(formatted)

# Build Runnable RAG Chain
rag_chain = (
    {"context": retriever | format_docs_with_metadata, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

def run_rag_query(query: str):
    docs = retriever.invoke(query)
    answer = rag_chain.invoke(query)
    return answer, docs

print("Local RAG Pipeline compiled and ready.")

Local RAG Pipeline compiled and ready.


In [5]:
print("=" * 70)
print("DEMONSTRATING HALLUCINATION HANDLING (OUT-OF-DOMAIN QUESTIONS)")
print("=" * 70)

out_of_domain_queries = [
    "What is the time complexity of the QuickSort algorithm?",
    "What was Apple's gross revenue in 2024?",
    "How do you configure a BGP router in Cisco IOS?"
]

for idx, query in enumerate(out_of_domain_queries, 1):
    answer, _ = run_rag_query(query)
    print(f"\n[Test Query {idx}]: '{query}'")
    print(f"-> Response: {answer}")

print("=" * 70)

DEMONSTRATING HALLUCINATION HANDLING (OUT-OF-DOMAIN QUESTIONS)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Test Query 1]: 'What is the time complexity of the QuickSort algorithm?'
-> Response: You are an academic assistant for an Organizational Behavior course.
Answer the question using ONLY the provided context below.

STRICT RULES:
1. If the required information is NOT available in the context, respond EXACTLY with:
   "I couldn't find this information in the provided knowledge base."
2. DO NOT use external knowledge, speculate, or invent details.

Context:
--- Document: OB.pdf (Page 53) ---
53
“benevolent autocrat.” Though they listen considerably to their
followers’ opinion before making any decision, the decision
remains to be their own.
They seemingly consider their subordinate’s ideas but when it
comes to decision making they are more autocratic than
benevolent. An advantage of autocratic leadership is the speed of
decision-making, as the leader does not have to obtain group
members approval.
Democratic or Participative Leadership
In contrast to autocratic leadership, democratic or

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Test Query 2]: 'What was Apple's gross revenue in 2024?'
-> Response: You are an academic assistant for an Organizational Behavior course.
Answer the question using ONLY the provided context below.

STRICT RULES:
1. If the required information is NOT available in the context, respond EXACTLY with:
   "I couldn't find this information in the provided knowledge base."
2. DO NOT use external knowledge, speculate, or invent details.

Context:
--- Document: OB.pdf (Page 20) ---
Most of the management experts have accepted only four
independent variables i.e. productivity, job satisfaction, employee
turnover and absenteeism. Let us brieﬂy study these factors:
a-Productivity: When we decide to study productivity, actually what
we want to know is the factors that would inﬂuence high
productivity. The study reveals that it is effectiveness and
eﬃciency of individual, groups and organizational structure that
have a positive impact on productivity. It should therefore be an
endeavour of any org

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Test Query 3]: 'How do you configure a BGP router in Cisco IOS?'
-> Response: You are an academic assistant for an Organizational Behavior course.
Answer the question using ONLY the provided context below.

STRICT RULES:
1. If the required information is NOT available in the context, respond EXACTLY with:
   "I couldn't find this information in the provided knowledge base."
2. DO NOT use external knowledge, speculate, or invent details.

Context:
--- Document: OB.pdf (Page 5) ---
the information, eﬃcient communication system prevailing in the
organization, well-deﬁned authority and responsibility supported
by detailed policies, rules and regulations.
The organization must have well laid out systems, which are
understood by workers, supervisors and managers. The leader
must keep open mind while dealing with subordinates and
exercise full control over various systems, levels and ensure
planned productivity and achieve high level of job satisfaction.
b-Technology

--- Document: OB.pdf (

In [8]:
eval_dataset = [
    {
        "id": 1,
        "question": "What is the definition of Organizational Behaviour according to Stephen P. Robbins?",
        "expected_source": "Page 2"
    },
    {
        "id": 2,
        "question": "What are the four basic managerial objectives in studying OB?",
        "expected_source": "Page 4"
    },
    {
        "id": 3,
        "question": "What were the main conclusions of Elton Mayo's Hawthorne studies?",
        "expected_source": "Page 9"
    },
    {
        "id": 4,
        "question": "What are McGregor's Theory X and Theory Y?",
        "expected_source": "Page 12"
    },
    {
        "id": 5,
        "question": "What is the capital city of Australia?",
        "expected_source": "N/A"
    }
]

print("=" * 70)
print("RUNNING LOCAL BENCHMARK")
print("=" * 70)

retrieval_hits = 0

for item in eval_dataset:
    q = item["question"]
    exp_src = item["expected_source"]
    
    answer, retrieved_docs = run_rag_query(q)
    retrieved_pages = [f"Page {d.metadata.get('page', 0) + 1}" for d in retrieved_docs]

    hit = (exp_src == "N/A") or any(exp_src in p for p in retrieved_pages)
    if hit:
        retrieval_hits += 1

    print(f"\n[Q{item['id']}] {q}")
    print(f"   Expected Source : {exp_src} | Retrieved Pages: {retrieved_pages}")
    print(f"   Retrieval Status: {'PASS' if hit else 'FAIL'}")
    print(f"   Generated Answer: {answer}")

print("\n" + "-" * 70)
print(f"Total Questions Evaluated : {len(eval_dataset)}")
print(f"Retrieval Hit Rate        : {retrieval_hits / len(eval_dataset) * 100:.1f}%")
print("=" * 70)

RUNNING LOCAL BENCHMARK


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Q1] What is the definition of Organizational Behaviour according to Stephen P. Robbins?
   Expected Source : Page 2 | Retrieved Pages: ['Page 13', 'Page 2', 'Page 2']
   Retrieval Status: PASS
   Generated Answer: You are an academic assistant for an Organizational Behavior course.
Answer the question using ONLY the provided context below.

STRICT RULES:
1. If the required information is NOT available in the context, respond EXACTLY with:
   "I couldn't find this information in the provided knowledge base."
2. DO NOT use external knowledge, speculate, or invent details.

Context:
--- Document: OB.pdf (Page 13) ---
13
McGregor in his theory displayed a total recognition of humanistic
approach. He laid stress on delegation of authority, setting
organizational objectives and leaving it to the employees to attain
the same. He carried forward the behaviour approach proposed by
Mayo, Maslow in mid. 20th century. McGregor’s theory Y is as valid
today as it was 50 years ago.
Chp2:Individual 

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (540 > 512). Running this sequence through the model will result in indexing errors



[Q2] What are the four basic managerial objectives in studying OB?
   Expected Source : Page 4 | Retrieved Pages: ['Page 20', 'Page 1', 'Page 8']
   Retrieval Status: FAIL
   Generated Answer: You are an academic assistant for an Organizational Behavior course.
Answer the question using ONLY the provided context below.

STRICT RULES:
1. If the required information is NOT available in the context, respond EXACTLY with:
   "I couldn't find this information in the provided knowledge base."
2. DO NOT use external knowledge, speculate, or invent details.

Context:
--- Document: OB.pdf (Page 20) ---
individuals. It helps to change the individual behavior. Values are
of a great help to determine as to what is good and
what is desirable. Values are so strong an instrument that it at
times ulter rational decisions. An organization having right value
system have greater chances of growth.
4-DEVELOPING AN ORGANIZATIONAL BEHAVIOUR MODEL (OB
MODEL)
Objective of model development is to understand h

...


[Q3] What were the main conclusions of Elton Mayo's Hawthorne studies?
   Expected Source : Page 9 | Retrieved Pages: ['Page 9', 'Page 10', 'Page 41']
   Retrieval Status: PASS
   Generated Answer: You are an academic assistant for an Organizational Behavior course.
Answer the question using ONLY the provided context below.

STRICT RULES:
1. If the required information is NOT available in the context, respond EXACTLY with:
   "I couldn't find this information in the provided knowledge base."
2. DO NOT use external knowledge, speculate, or invent details.

Context:
--- Document: OB.pdf (Page 9) ---
3. Group standards established individual output.
4. Money was less a factor in determining output.
5. Group standards, group sentiments and security provided by the
group were responsible for higher productivity.
Elton Mayo made a new beginning on human factor in functioning
of the organization and attainment of organizational goals.
Hawthorne studies have proved that experiment and behavio

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Q4] What are McGregor's Theory X and Theory Y?
   Expected Source : Page 12 | Retrieved Pages: ['Page 12', 'Page 12', 'Page 13']
   Retrieval Status: PASS
   Generated Answer: You are an academic assistant for an Organizational Behavior course.
Answer the question using ONLY the provided context below.

STRICT RULES:
1. If the required information is NOT available in the context, respond EXACTLY with:
   "I couldn't find this information in the provided knowledge base."
2. DO NOT use external knowledge, speculate, or invent details.

Context:
--- Document: OB.pdf (Page 12) ---
12
Theory X and Theory Y were introduced by Douglas McGregor
based on two diagonally opposite views of human behaviour.
Theory X
McGregor assumed that average human being dislikes work and
will try to avoid it if possible. Employees are lazy, they must be
controlled, coerced and even punished to achieve organizational
goals. Average employee do not accept responsibility and seek
direction from their superiors. 

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Q5] What is the capital city of Australia?
   Expected Source : N/A | Retrieved Pages: ['Page 39', 'Page 55', 'Page 38']
   Retrieval Status: PASS
   Generated Answer: You are an academic assistant for an Organizational Behavior course.
Answer the question using ONLY the provided context below.

STRICT RULES:
1. If the required information is NOT available in the context, respond EXACTLY with:
   "I couldn't find this information in the provided knowledge base."
2. DO NOT use external knowledge, speculate, or invent details.

Context:
--- Document: OB.pdf (Page 39) ---
necessary facilities as per Factories Act. Light, ventilation,
cleanliness, enough space for work, immediate availability of
supervision, adequate latest tools and generally good surrounding

--- Document: OB.pdf (Page 55) ---
55

--- Document: OB.pdf (Page 38) ---
(c) Job satisfaction is related to job dimensions. These can be
expressed in terms of job content, remuneration, attitude of co-
workers, and opportunity of

In [ ]:
def gradio_respond(user_question: str):
    if not user_question.strip():
        return "Please enter a valid question.", "No context retrieved."

    answer, retrieved_docs = run_rag_query(user_question)

    sources_markdown = ""
    for idx, doc in enumerate(retrieved_docs, start=1):
        source_file = os.path.basename(doc.metadata.get("source", "OB.pdf"))
        page_num = doc.metadata.get("page", 0) + 1
        excerpt = doc.page_content.replace("\n", " ").strip()

        sources_markdown += f"### Source {idx}\n"
        sources_markdown += f"- **Document**: `{source_file}`\n"
        sources_markdown += f"- **Location**: Page {page_num}\n"
        sources_markdown += f"- **Context Excerpt**: *\"{excerpt[:250]}...\"*\n\n---\n"

    return answer, sources_markdown

# Build Gradio UI
with gr.Blocks(title="OB Academic Assistant (Local)") as demo:
    gr.Markdown("# 📚 Local Organizational Behavior Academic RAG Assistant")
    gr.Markdown(
        "Powered by **Sentence-Transformers** and **Flan-T5** (100% Local / Zero API Keys)."
    )

    with gr.Row():
        with gr.Column(scale=2):
            question_input = gr.Textbox(
                lines=2,
                placeholder="e.g., What are McGregor's Theory X and Theory Y?",
                label="Ask a Question"
            )
            submit_btn = gr.Button("Submit Question", variant="primary")

            gr.Examples(
                examples=[
                    "What is the definition of Organizational Behaviour according to Stephen P. Robbins?",
                    "What are the four basic managerial objectives in studying OB?",
                    "What were the main conclusions of Elton Mayo's Hawthorne studies?",
                    "What are McGregor's Theory X and Theory Y?",
                    "What is the time complexity of the QuickSort algorithm?"
                ],
                inputs=question_input
            )

        with gr.Column(scale=3):
            answer_output = gr.Textbox(lines=6, label="Generated Answer", interactive=False)
            sources_output = gr.Markdown(label="Retrieved Sources & Context")

    submit_btn.click(
        fn=gradio_respond,
        inputs=[question_input],
        outputs=[answer_output, sources_output]
    )
    question_input.submit(
        fn=gradio_respond,
        inputs=[question_input],
        outputs=[answer_output, sources_output]
    )

# Launch Gradio server
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://0b92512e142e3aff26.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (535 > 512). Running this sequence through the model will result in indexing errors
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
